# Sutton & Barto Chapter 2: 2.8
## Gradient Bandit Algorithms

This notebook gives a minimal implementation of the **gradient bandit** method from Sutton and Barto Chapter 2.8.

Instead of learning action values directly, the algorithm learns **preferences** over actions and turns them into a stochastic policy with softmax.

## What You Will See

- action preferences instead of direct action-value estimates
- softmax action probabilities
- the gradient bandit update with and without a baseline
- a compact experiment on the 10-armed testbed

Dependencies: `numpy`, `matplotlib`

## Theory

Let $H_t(a)$ be the preference for action $a$. The policy is defined by softmax:

$$
\pi_t(a) = \frac{e^{H_t(a)}}{\sum_b e^{H_t(b)}}.
$$

After selecting action $A_t$ and observing reward $R_t$, we update preferences in the direction of higher expected reward. With a baseline $\bar{R}_t$, the update is

$$
H_{t+1}(A_t) = H_t(A_t) + \alpha (R_t - \bar{R}_t)(1 - \pi_t(A_t)),
$$

and for all other actions $a \neq A_t$,

$$
H_{t+1}(a) = H_t(a) - \alpha (R_t - \bar{R}_t)\pi_t(a).
$$

The baseline reduces variance and usually makes learning more stable.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8')

In [ ]:
class TenArmedBandit:
    def __init__(self, k=10, mean=4.0, seed=None):
        self.k = k
        self.rng = np.random.default_rng(seed)
        self.q_true = self.rng.normal(mean, 1.0, size=k)

    def step(self, action):
        reward = self.rng.normal(self.q_true[action], 1.0)
        optimal = int(action == np.argmax(self.q_true))
        return reward, optimal


def softmax(x):
    z = x - np.max(x)
    exp_z = np.exp(z)
    return exp_z / np.sum(exp_z)

In [ ]:
def run_gradient_bandit(steps=1000, alpha=0.1, use_baseline=True, seed=None):
    bandit = TenArmedBandit(seed=seed)
    rng = np.random.default_rng(seed)
    preferences = np.zeros(bandit.k)
    rewards = np.zeros(steps)
    optimal_actions = np.zeros(steps)
    avg_reward = 0.0

    for t in range(steps):
        probs = softmax(preferences)
        action = rng.choice(bandit.k, p=probs)
        reward, optimal = bandit.step(action)
        baseline = avg_reward if use_baseline else 0.0
        one_hot = np.zeros(bandit.k)
        one_hot[action] = 1.0
        preferences += alpha * (reward - baseline) * (one_hot - probs)
        avg_reward += (reward - avg_reward) / (t + 1)

        rewards[t] = reward
        optimal_actions[t] = optimal

    return rewards, optimal_actions


def average_runs(runs=200, **kwargs):
    reward_history = []
    optimal_history = []
    for seed in range(runs):
        rewards, optimal = run_gradient_bandit(seed=seed, **kwargs)
        reward_history.append(rewards)
        optimal_history.append(optimal)
    return np.mean(reward_history, axis=0), 100 * np.mean(optimal_history, axis=0)

## Experiment

The textbook often compares gradient bandit variants on a testbed where the true action values have positive mean. We keep that setup by sampling $q_*(a) \sim \mathcal{N}(4, 1)$.

Below we compare the same algorithm with and without the average-reward baseline.

In [ ]:
reward_base, opt_base = average_runs(runs=200, steps=1000, alpha=0.1, use_baseline=True)
reward_no_base, opt_no_base = average_runs(runs=200, steps=1000, alpha=0.1, use_baseline=False)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(reward_base, label='with baseline')
ax[0].plot(reward_no_base, label='without baseline')
ax[0].set_title('Average Reward')
ax[0].set_xlabel('Steps')
ax[0].set_ylabel('Average reward')
ax[0].legend()

ax[1].plot(opt_base, label='with baseline')
ax[1].plot(opt_no_base, label='without baseline')
ax[1].set_title('% Optimal Action')
ax[1].set_xlabel('Steps')
ax[1].set_ylabel('% optimal action')
ax[1].legend()
plt.tight_layout()

## Takeaway

- Gradient bandits optimize a **policy** directly instead of estimating action values first.
- Softmax preferences guarantee ongoing exploration because every action keeps some probability mass.
- The average-reward baseline usually improves learning by lowering update variance.

## Reference

Richard S. Sutton and Andrew G. Barto, *Reinforcement Learning: An Introduction*, Chapter 2.